In [ ]:
import sqlite3
import pandas as pd

In [ ]:
# File paths
CSV_FILE = "../../data/processed/processed_data.csv"
DB_FILE = "../../data/database.db"
TABLE_NAME = "records"

In [ ]:
# Connect to SQLite
conn = sqlite3.connect(DB_FILE)

In [ ]:
# Drop table if already exists
cursor = conn.cursor()
cursor.execute(f"DROP TABLE IF EXISTS {TABLE_NAME}")
conn.commit()

In [ ]:
# Read CSV in chunks to handle large files
chunksize = 100_000
for i, chunk in enumerate(pd.read_csv(CSV_FILE, chunksize=chunksize)):
    print(f"Processing chunk {i+1}...")
    chunk.to_sql(TABLE_NAME, conn, if_exists="append", index=False)

# Create index for fast lookup on Customer
cursor = conn.cursor()
cursor.execute(f"CREATE INDEX IF NOT EXISTS idx_customer ON {TABLE_NAME}(Customer)")
cursor.execute(f"CREATE INDEX IF NOT EXISTS idx_merchant ON {TABLE_NAME}(Merchant)")
conn.commit()

In [ ]:
# Close connection
conn.close()
print("Database created successfully.")

In [ ]:
def get_last_entry(filters):
    conn = sqlite3.connect(DB_FILE)

    # Build WHERE clause dynamically
    where_clause = " AND ".join([f"{col} = ?" for col in filters.keys()]) if filters else "1=1"
    values = tuple(filters.values())

    query = f"""
        SELECT *
        FROM {TABLE_NAME}
        WHERE {where_clause}
        ORDER BY rowid DESC
        LIMIT 1
    """

    df = pd.read_sql_query(query, conn, params=values)
    conn.close()
    return df

In [ ]:
# read latest entry for customer
filters = {"Customer" : 'C1166355595', "Merchant": "M348934600"}
cust_row = get_last_entry(filters)

cust_row